# 第4章：Implementing a GPT Model from Scratch

**目标：** 从零搭建完整的 GPT 模型架构，并实现简单的文本生成

```
DummyGPT(整体骨架) → LayerNorm → GELU + FeedForward → 残差连接 → TransformerBlock → GPTModel → 文本生成
```

**前置回顾（第2、3章）：**
- 文本经过 BPE 分词 → Token Embedding + Positional Embedding → 输入向量 `(batch, seq_len, emb_dim)`
- Multi-Head Causal Attention 让每个 token 融合上下文信息，输出 shape 不变
- 现在的问题：光有注意力还不够，还需要 **LayerNorm、FeedForward、残差连接** 才能组成完整的 Transformer Block，多个 Block 堆叠才构成 GPT

---

## 4.1 GPT 架构概览 ⭐

GPT 属于 **decoder-only** 架构（只用了原始 Transformer 的解码器部分）。

**GPT-2 (124M) 的核心配置：**

| 参数 | 值 | 含义 |
|------|-----|------|
| `vocab_size` | 50,257 | BPE 词汇表大小 |
| `context_length` | 1,024 | 最大输入 token 数 |
| `emb_dim` | 768 | 嵌入维度 |
| `n_heads` | 12 | 注意力头数 |
| `n_layers` | 12 | Transformer Block 层数 |
| `drop_rate` | 0.1 | Dropout 比率 |
| `qkv_bias` | False | Q/K/V 投影是否带偏置 |

**整体数据流：**
```
Token IDs
   ↓
Token Embedding + Position Embedding
   ↓ Dropout
┌─────────────────────────────┐
│     Transformer Block × 12  │
│  ┌─────────────────────┐    │
│  │ LayerNorm            │    │
│  │ Multi-Head Attention  │    │
│  │ Dropout + Residual   │    │
│  │ LayerNorm            │    │
│  │ FeedForward          │    │
│  │ Dropout + Residual   │    │
│  └─────────────────────┘    │
└─────────────────────────────┘
   ↓
Final LayerNorm
   ↓
Linear (output head) → logits (vocab_size)
```

> 💡 **关键洞察：** GPT 架构虽然有 1.24 亿参数，但代码量并不大——大量元素是**重复堆叠**的。理解一个 Transformer Block，就理解了整个模型。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

print("GPT-2 124M 配置:")
for k, v in GPT_CONFIG_124M.items():
    print(f"  {k}: {v}")

### 先搭骨架：DummyGPT

我们先用占位符（Dummy）组件搭出 GPT 的整体骨架，确认数据流正确后，再逐步替换为真实实现。

In [ ]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
# 测试 DummyGPT 的数据流
tokenizer = tiktoken.get_encoding("gpt2")

batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)

torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)

print(f"输入 batch shape: {batch.shape}")        # (2, 4)
print(f"输出 logits shape: {logits.shape}")       # (2, 4, 50257)
print(f"\n→ 每个 token 位置输出一个 {GPT_CONFIG_124M['vocab_size']} 维向量")
print(f"→ 代表该位置预测下一个 token 的 logits（未归一化概率）")

> 💡 **关键洞察：** DummyGPT 虽然 TransformerBlock 和 LayerNorm 都是空壳，但数据流已经完全正确：
> - 输入 `(batch, seq_len)` 的 token IDs
> - 输出 `(batch, seq_len, vocab_size)` 的 logits
> 
> 接下来我们要做的就是**逐一替换**每个 Dummy 组件为真实实现。

---
## 4.2 Layer Normalization ⭐

**核心问题：** 深度网络训练时，每层输入的分布会不断变化（internal covariate shift），导致训练不稳定。

**LayerNorm 的做法：** 对每个样本的**特征维度**做归一化（均值→0，方差→1），然后加上可学习的缩放（scale）和偏移（shift）。

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

其中 $\mu$ 和 $\sigma^2$ 是沿最后一维计算的均值和方差，$\gamma$（scale）和 $\beta$（shift）是可学习参数。

**为什么是 LayerNorm 而不是 BatchNorm？**
- BatchNorm 沿 batch 维度归一化，在 NLP 中效果差（序列长度不一，batch 小时统计量不稳定）
- LayerNorm 沿特征维度归一化，**每个样本独立**，不依赖 batch 中的其他样本

In [ ]:
# 先直观理解归一化的效果
torch.manual_seed(123)
batch_example = torch.randn(2, 5)  # 2个样本, 5个特征

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(f"层输出:\n{out}")

# 沿特征维度计算均值和方差
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print(f"\n均值: {mean.squeeze()}")
print(f"方差: {var.squeeze()}")
print("→ 不同样本的均值和方差差异较大")

In [ ]:
# 手动归一化
out_norm = (out - mean) / torch.sqrt(var)
mean_after = out_norm.mean(dim=-1, keepdim=True)
var_after = out_norm.var(dim=-1, keepdim=True)

torch.set_printoptions(sci_mode=False)
print(f"归一化后均值: {mean_after.squeeze()}")
print(f"归一化后方差: {var_after.squeeze()}")
print("→ 均值≈0, 方差≈1")

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
# 验证我们的 LayerNorm
ln = LayerNorm(emb_dim=6)
out_ln = ln(out)

mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print(f"LayerNorm 后均值: {mean.squeeze()}")
print(f"LayerNorm 后方差: {var.squeeze()}")
print("\n→ 方差不是精确的 1.0，因为加了 eps 防止除零")

### 🤔 思考：LayerNorm 的两个设计细节

| 设计 | 原因 |
|------|------|
| `scale` 和 `shift` 可学习 | 纯归一化可能丢失有用信息，scale/shift 让模型自己决定最佳分布 |
| `unbiased=False`（有偏方差） | GPT-2 原版使用有偏方差（除以 n 而非 n-1），对于 768 维来说差异极小，但为了兼容预训练权重而保持一致 |

> 💡 **GPT 中 LayerNorm 的位置（Pre-LN）：** GPT-2 采用 **Pre-LN** 架构——LayerNorm 放在注意力/FFN **之前**，而非之后。这种设计使训练更稳定，是目前 LLM 的标准做法。

### ✏️ 练习
1. `eps` 设为 `1e-5` 的作用是什么？如果去掉会怎样？
2. 如果把 `unbiased=False` 改成 `True`，结果会有多大差异？试试在 `emb_dim=768` 时对比两种方式的输出。

In [ ]:
# 在这里做实验

---
## 4.3 GELU 激活函数与 FeedForward 网络 ⭐

Transformer Block 中，注意力层之后紧跟一个 **前馈网络（FeedForward Network, FFN）**。

FFN 的结构很简单：**两层线性变换夹一个激活函数**。

```
输入 (emb_dim=768)
   ↓ Linear(768, 3072)    ← 扩展 4 倍
   ↓ GELU 激活
   ↓ Linear(3072, 768)    ← 压缩回原维度
输出 (emb_dim=768)
```

### 为什么用 GELU 而不是 ReLU？

| | ReLU | GELU |
|---|---|---|
| 公式 | $\max(0, x)$ | $x \cdot \Phi(x)$ ，$\Phi$ 是标准正态 CDF |
| 特点 | 分段线性，x<0 时梯度为 0 | 平滑，x<0 时梯度不为零 |
| 优势 | 计算快 | 更好的梯度流动，LLM 训练效果更好 |

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
# 可视化 GELU vs ReLU
gelu, relu = GELU(), nn.ReLU()
x = torch.linspace(-3, 3, 100)

plt.figure(figsize=(8, 3))
for i, (fn, label) in enumerate(zip([gelu, relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, fn(x))
    plt.title(f"{label} 激活函数")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)
plt.tight_layout()
plt.show()

print("→ GELU 是平滑的，负数区域有微小非零输出")
print("→ ReLU 是分段线性的，负数区域直接截断为 0")

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# 验证 FeedForward 的输入输出形状
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 4, 768)  # (batch, seq_len, emb_dim)
out = ffn(x)

print(f"FFN 输入 shape: {x.shape}")
print(f"FFN 输出 shape: {out.shape}")
print(f"\n→ 输入输出 shape 相同，中间经历 768→3072→768 的维度变化")

# 统计参数量
ffn_params = sum(p.numel() for p in ffn.parameters())
print(f"\nFFN 参数量: {ffn_params:,}")
print(f"  第一层: 768×3072 + 3072(bias) = {768*3072 + 3072:,}")
print(f"  第二层: 3072×768 + 768(bias) = {3072*768 + 768:,}")

> 💡 **关键洞察：为什么中间层要扩大 4 倍？**
>
> FFN 的 768→3072→768 设计被称为 **bottleneck**（瓶颈结构的反面——其实是**expansion**）。
> 扩大 4 倍可以让模型在更高维空间中做非线性变换，然后再压缩回原始维度。
> 这个 4 倍的比例在原始 Transformer 论文中就确定了，后来的 GPT、BERT 等都沿用了。

### ✏️ 练习
1. 计算一下：12 层 Transformer Block 中，所有 FFN 的总参数量是多少？占 GPT-2 124M 总参数的百分比？
2. 把 GELU 替换成 ReLU，对比一下 FFN 的输出有什么不同？

In [ ]:
# 在这里做实验

---
## 4.4 残差连接（Shortcut / Skip Connection）⭐⭐

**核心问题：** 深度网络中，梯度在反向传播时可能逐层衰减，导致**梯度消失**——前面的层几乎学不到东西。

**残差连接的做法：** 跳过中间层，将输入直接加到输出上。

```
没有残差连接:    x → Layer → y
有残差连接:      x → Layer → y + x    ← 加上原始输入
```

**为什么有效？**
- 梯度可以通过 shortcut 直接传到前面的层（不需要经过所有中间层）
- 即使中间层输出接近 0，信号仍然可以通过

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x


def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss()(output, target)
    loss.backward()
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name}: 梯度绝对值均值 = {param.grad.abs().mean().item():.6f}")

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])

print("=" * 60)
print("【无残差连接】")
print("=" * 60)
torch.manual_seed(123)
model_no_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)
print_gradients(model_no_shortcut, sample_input)

print("\n" + "=" * 60)
print("【有残差连接】")
print("=" * 60)
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print_gradients(model_with_shortcut, sample_input)

> 💡 **关键洞察：** 对比两种情况可以看到：
> - **无残差连接：** 越靠前的层（layer.0）梯度越小 → 梯度消失
> - **有残差连接：** 所有层的梯度都在合理范围内 → 训练更均匀
>
> 这就是为什么 Transformer 每个子层（注意力、FFN）都要加残差连接。

### ✏️ 练习
1. 如果把层数从 5 增加到 10 或 20，梯度消失的现象会更严重吗？用代码验证。
2. 残差连接要求 `x.shape == layer_output.shape`，如果两者 shape 不同怎么办？实际中是怎么处理的？

In [ ]:
# 在这里做实验

---
## 4.5 Transformer Block ⭐⭐

现在我们把所有组件组合在一起，构建 **Transformer Block**——GPT 的基本构建单元。

**一个 Transformer Block 的结构：**
```
输入 x
  │
  ├──────────────────────────┐ (残差连接)
  ↓                          │
LayerNorm                    │
  ↓                          │
Multi-Head Causal Attention  │
  ↓                          │
Dropout                      │
  ↓                          │
  + ←─────────────────────────┘ (加上原始输入)
  │
  ├──────────────────────────┐ (残差连接)
  ↓                          │
LayerNorm                    │
  ↓                          │
FeedForward                  │
  ↓                          │
Dropout                      │
  ↓                          │
  + ←─────────────────────────┘ (加上原始输入)
  │
输出
```

注意：这里我们需要用到第 3 章实现的 `MultiHeadAttention`，直接在这里重新定义。

In [ ]:
# 第3章实现的 MultiHeadAttention（高效版本）
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out 必须能被 num_heads 整除"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(
            self.mask[:num_tokens, :num_tokens].bool(),
            -torch.inf
        )
        attn_weights = F.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = attn_weights @ values
        context_vecs = context_vecs.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        context_vecs = self.out_proj(context_vecs)

        return context_vecs

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # 子层 1: Multi-Head Attention + 残差连接
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # 子层 2: FeedForward + 残差连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [ ]:
# 验证 TransformerBlock
torch.manual_seed(123)
x = torch.rand(2, 4, 768)  # (batch=2, seq_len=4, emb_dim=768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print(f"输入 shape: {x.shape}")
print(f"输出 shape: {output.shape}")
print(f"\n→ 输入输出 shape 完全相同！这保证了 Block 可以堆叠")

# TransformerBlock 参数量
block_params = sum(p.numel() for p in block.parameters())
print(f"\n单个 TransformerBlock 参数量: {block_params:,}")
print(f"12 个 Block 总参数量: {block_params * 12:,}")

### 🤔 思考：Pre-LN vs Post-LN

| | Post-LN（原始Transformer） | Pre-LN（GPT-2）|
|---|---|---|
| LN 位置 | 注意力/FFN **之后** | 注意力/FFN **之前** |
| 残差连接 | `x + LN(SubLayer(x))` | `x + SubLayer(LN(x))` |
| 训练稳定性 | 需要 warmup | 更稳定，不一定需要 warmup |
| 实际使用 | 原始 Transformer, BERT | GPT-2, GPT-3, LLaMA 等 |

> 💡 我们实现的是 **Pre-LN** 版本（先 Norm 再做子层运算），这是 GPT-2 及后续大模型的标准做法。

### ✏️ 练习
1. 验证：如果把 `norm1` 移到 `self.att(x)` 之后（Post-LN），输出 shape 是否还正确？
2. 单个 TransformerBlock 中，MHA 和 FFN 各占多少参数？哪个更多？

In [ ]:
# 在这里做实验

---
## 4.6 组装完整 GPTModel ⭐⭐⭐

现在把所有组件组装到一起！用真实的 `TransformerBlock` 和 `LayerNorm` 替换之前的 Dummy 版本。

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
# 实例化真正的 GPT 模型
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print(f"输入 batch shape: {batch.shape}")
print(f"输出 logits shape: {out.shape}")
print(f"\n→ 与 DummyGPT 的输出 shape 完全一致")
print(f"→ 但现在每个 Transformer Block 都是真实的计算")

In [ ]:
# 参数量分析
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

print(f"\n→ 为什么是 ~163M 而不是 124M？")
print(f"\nToken Embedding 层 shape: {model.tok_emb.weight.shape}")
print(f"Output 层 shape:          {model.out_head.weight.shape}")
print(f"→ 两者 shape 一样！都是 (50257, 768)")

# GPT-2 原版使用 weight tying（共享这两个矩阵）
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"\n去掉 output 层后（weight tying）: {total_params_gpt2:,}")
print(f"→ 这就是 124M 参数的来源！")

In [ ]:
# 模型大小（显存占用）
total_size_bytes = total_params * 4  # float32: 4 bytes per parameter
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"模型大小 (float32): {total_size_mb:.2f} MB")
print(f"模型大小 (float16): {total_size_mb / 2:.2f} MB")

### 🤔 思考：Weight Tying（权重共享）

GPT-2 原版将 Token Embedding 和 Output Head 共享权重：`self.out_head.weight = self.tok_emb.weight`

**直觉理解：**
- Token Embedding 的工作：token ID → 语义向量
- Output Head 的工作：语义向量 → 预测哪个 token（本质是反向映射）
- 两者互为「逆操作」，共享权重是合理的

**我们没有实现 weight tying 的原因：**
- 不共享训练效果可能更好
- 第 5 章加载 GPT-2 预训练权重时会重新讨论这个话题

**GPT-2 各版本参数对比：**

| 版本 | emb_dim | n_layers | n_heads | 参数量 |
|------|---------|----------|---------|--------|
| GPT2-small | 768 | 12 | 12 | 124M |
| GPT2-medium | 1024 | 24 | 16 | 345M |
| GPT2-large | 1280 | 36 | 20 | 762M |
| GPT2-XL | 1600 | 48 | 25 | 1542M |

### ✏️ 练习
1. 修改配置为 GPT2-medium（emb_dim=1024, n_layers=24, n_heads=16），计算参数量，验证是否接近 345M。
2. GPT-2 的 `qkv_bias=False`，但加载预训练权重时会设为 `True`。为什么？（提示：OpenAI 原始实现用了 bias）

In [ ]:
# 在这里做实验

---
## 4.7 文本生成（Greedy Decoding）⭐⭐

模型搭好了，虽然还没训练，但我们可以先实现**文本生成**的逻辑。

**Greedy Decoding（贪心解码）：**
- 每一步选择概率最高的 token 作为输出
- 简单快速，但容易重复、不够多样

```
输入: "Hello, I am"
   ↓ 模型预测下一个 token 的概率分布
选择概率最高的 token (argmax)
   ↓ 将新 token 追加到输入
"Hello, I am [new_token]"
   ↓ 重复...
"Hello, I am [new_token] [new_token2] ..."
```

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """简单的贪心解码文本生成"""
    for _ in range(max_new_tokens):
        # 截断到上下文长度
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        # 只取最后一个位置的预测
        logits = logits[:, -1, :]  # (batch, vocab_size)

        # 贪心: 选概率最高的 token
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # 追加到序列
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
# 用未训练的模型生成文本
model.eval()

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # (1, n_tokens)

print(f"输入文本: '{start_context}'")
print(f"Token IDs: {encoded}")
print(f"输入 shape: {encoded_tensor.shape}")

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"],
)

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(f"\n生成结果: '{decoded_text}'")
print(f"\n→ 输出是随机的，因为模型还没有训练！")
print(f"→ 第 5 章会训练这个模型，届时会生成有意义的文本")

### 🤔 思考：为什么只用最后一个位置的 logits？

模型输出 `(batch, seq_len, vocab_size)`，但我们只取 `logits[:, -1, :]`（最后一个位置），为什么？

因为 GPT 是**因果模型**：
- 位置 0 的输出只看到了 token 0 → 预测 token 1
- 位置 1 的输出看到了 token 0, 1 → 预测 token 2
- ...
- 位置 n-1 的输出看到了 token 0..n-1 → 预测 token n

所以**最后一个位置**的输出融合了所有已知信息，是预测下一个 token 的最佳位置。

### ✏️ 练习
1. 增加 `max_new_tokens` 到 20 或 50，观察生成结果有什么特点？
2. 如果输入文本长度超过 `context_length`（1024 tokens），`generate_text_simple` 是如何处理的？这种处理方式有什么缺点？
3. （进阶）修改 `generate_text_simple`，把 `argmax` 换成按概率采样（`torch.multinomial`），观察生成结果的变化。

In [ ]:
# 在这里做实验

---
## 4.8 完整流程回顾

让我们用一张图回顾整个第4章的内容：

In [ ]:
print("═" * 70)
print("  第4章完整流程：从零搭建 GPT 模型")
print("═" * 70)
print()
print("  1. DummyGPT ─────────── 搭建骨架，确认数据流")
print("     ↓")
print("  2. LayerNorm ────────── 归一化：均值→0, 方差→1")
print("     ↓")
print("  3. GELU + FeedForward ─ 非线性变换：768→3072→768")
print("     ↓")
print("  4. 残差连接 ──────────── 防止梯度消失")
print("     ↓")
print("  5. TransformerBlock ─── LN + MHA + 残差 + LN + FFN + 残差")
print("     ↓")
print("  6. GPTModel ─────────── 12 个 Block 堆叠 → 完整 GPT-2 (124M)")
print("     ↓")
print("  7. 文本生成 ──────────── Greedy Decoding (尚未训练)")
print("     ↓")
print("  8. 第5章：预训练 ────── 让模型真正学会语言！ 🚀")
print()
print("═" * 70)
print(f"\n  模型配置: GPT-2 124M")
print(f"  总参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"  模型大小: {sum(p.numel() for p in model.parameters()) * 4 / 1024 / 1024:.1f} MB (float32)")
print(f"  Transformer Blocks: {GPT_CONFIG_124M['n_layers']}")
print(f"  注意力头数: {GPT_CONFIG_124M['n_heads']}")
print(f"  嵌入维度: {GPT_CONFIG_124M['emb_dim']}")
print(f"  上下文长度: {GPT_CONFIG_124M['context_length']}")

---
## 📝 本章核心 Checklist

学完后确认你能回答：

- [ ] GPT 的整体架构是怎样的？数据是如何流经各个组件的？
- [ ] LayerNorm 的作用是什么？Pre-LN 和 Post-LN 有什么区别？
- [ ] GELU 激活函数与 ReLU 的区别？为什么 LLM 更倾向于用 GELU？
- [ ] FeedForward 为什么要先扩大 4 倍再缩回来？
- [ ] 残差连接如何缓解梯度消失问题？
- [ ] TransformerBlock 内部的两个子层分别是什么？各自的作用？
- [ ] GPT-2 124M 的参数量为什么实际是 ~163M？什么是 weight tying？
- [ ] Greedy Decoding 是如何工作的？为什么只取最后一个位置的 logits？
- [ ] 能否独立写出 GPTModel 的完整代码（不看笔记）？

全部能回答 → 进入第 5 章：在无标签数据上预训练 GPT！🚀